In [1]:
from openai import OpenAI

openai_client = OpenAI()

In [2]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

parsed_docs = [doc.parse() for doc in files]

chunked_docs = chunk_documents(
    parsed_docs,
    size=3000,
    step=1500
)

In [7]:
from minsearch import AppendableIndex

In [8]:
index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)

index.fit(chunked_docs)

In [9]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [10]:
import json

RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)

    return RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

In [11]:
question = "How do I create a dahsbord in Evidently?"

search_results = search(question)
user_prompt = build_prompt(question, search_results)

In [12]:
user_prompt = build_prompt(question, search_results)

In [13]:
messages = [
    {"role": "system", "content": RAG_INSTRUCTIONS},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=messages,
)

In [14]:
print(response.output_text)

The provided context does not contain specific instructions on how to create a dashboard in Evidently. Therefore, I cannot provide an answer to your question based solely on the given information.


In [15]:
instructions = """
You're a documentation assistant.

Answer the user question using the documentation knowledge base.

Use only facts from the knowledge base when answering.
IMPORTANT: If you cannot find the answer, inform the user.
"""

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": ["query"]
    }
}

In [16]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=messages,
    tools=[search_tool],
)

In [17]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_KJcFLDtmPwvfL0dQXVWYwgcO', name='search', type='function_call', id='fc_07975dd976bf56c7006ac7ba9deeb087d18941fef06a044ce7', async_=None, caller=None, namespace=None, status='completed')]

In [18]:
tool_call = response.output[0]

tool_call

ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_KJcFLDtmPwvfL0dQXVWYwgcO', name='search', type='function_call', id='fc_07975dd976bf56c7006ac7ba9deeb087d18941fef06a044ce7', async_=None, caller=None, namespace=None, status='completed')

In [19]:
tool_call.arguments

'{"query":"create dashboard in Evidently"}'

In [20]:
arguments = json.loads(tool_call.arguments)

arguments

{'query': 'create dashboard in Evidently'}

In [21]:
search_results = search(**arguments)

search_results[:1]

[{'start': 0,
  'content': "<Check>\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\n## What is a Dashboard?\n\nA Dashboard provides a clear view of your AI application performance. You can use it:\n\n- to track evaluation results across multiple experiments;\n- to track live production quality over time.\n\nEach Project has its own Dashboard. It's empty at first.\n\nTo populate it, you need to run an evaluation and **save at least one Report** to the Project. You can then choose values from Reports to plot.\n\n![](/images/dashboard_llm_light.png)\n\n## Dashboard Tabs\n\n<Check>\n  Multiple Tabs are available in **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\nYou can logically organize Panels within the same Dashboard into different Tabs.\n\n![](/images/dashboard_llm_tabs.gif)\n\n## **Dashboard Panels**\n\nA Panel is a visual element in the Dashboard that displays specified values in a single widget. Panels ca

In [22]:
messages.append(tool_call)

call_output = {
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": json.dumps(search_results),
}

messages.append(call_output)

In [23]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=messages,
    tools=[search_tool],
)

In [24]:
print(response.output_text)

To create a dashboard in Evidently, follow these steps:

1. **Ensure Prerequisites**:
   - **Service Availability**: Dashboards are available in **Evidently OSS**, **Evidently Cloud**, and **Evidently Enterprise**.
   - Create an account and log into Evidently Cloud if using that service.

2. **Create a Project**: Set up a new project where you will manage your dashboard.

3. **Run Evaluations**: The dashboard requires reports as data sources. Run an evaluation and save at least one report to your project.

4. **Navigate to Dashboard**: Each project has its own dashboard which starts empty.

5. **Add Panels**:
   - You can add panels using:
     - **Python API**: For programmatic control, you can define your dashboard as code.
     - **User Interface**: For direct interaction (available in Cloud and Enterprise).
   - **Panels** represent visual elements (like counters or plots) that display metrics. When adding a panel, specify:
     - **Value**: Choose a metric to plot.
     - **Param

In [25]:
def make_call(tool_call):
    function_name = tool_call.name
    arguments = json.loads(tool_call.arguments)

    if function_name == "search":
        result = search(**arguments)
    else:
        result = f"Unknown tool: {function_name}"

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [36]:
tools = [search_tool, add_entry_tool]

In [33]:
def loop(messages):
    while True:
        response = openai_client.responses.create(
            model="gpt-4o-mini",
            input=messages,
            tools=tools,
        )

        messages += response.output

        tool_calls = [
            item for item in response.output
            if item.type == "function_call"
        ]

        if not tool_calls:
            return response

        for tool_call in tool_calls:
            tool_result = make_call(tool_call)
            messages.append(tool_result)

In [27]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = loop(messages)

print(response.output_text)

To create a dashboard in Evidently, follow these steps:

### 1. Ensure Reports are Available
Before creating a dashboard, make sure you have run evaluations and saved at least one report to your project. This report serves as the data source for your dashboard.

### 2. Create the Dashboard
- **For No-code GUI in Evidently Cloud/Enterprise:**
  1. Log into your Evidently account.
  2. Create a new **Project** if you haven't done so.
  3. Enter "Edit" mode on the Dashboard.
  4. Click on "Add Tab" to organize panels. You can create custom tabs or use pre-built templates.
  5. Click "Add Panel" to configure and add various types of visual elements (e.g., counters, line plots).

- **Using Python API:**
  1. Ensure you have connected to Evidently and created a project.
  2. Use the following Python commands:
     ```python
     from evidently.sdk.models import PanelMetric
     from evidently.sdk.panels import DashboardPanelPlot
     
     project.dashboard.add_panel(
         DashboardPanel

In [28]:
def add_entry(title, description, content, filename):
    entry = {
        "title": title,
        "description": description,
        "content": content,
        "filename": filename,
    }

    index.append(entry)

    return entry
    

In [29]:
add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new entry to the documentation index.",
    "parameters": {
        "type": "object",
        "properties": {
            "title": {"type": "string"},
            "description": {"type": "string"},
            "content": {"type": "string"},
            "filename": {"type": "string"},
        },
        "required": ["title", "description", "content", "filename"],
    },
}

In [30]:
tools = [search_tool, add_entry_tool]

In [31]:
def make_call(tool_call):
    function_name = tool_call.name
    arguments = json.loads(tool_call.arguments)

    if function_name == "search":
        result = search(**arguments)
    elif function_name == "add_entry":
        result = add_entry(**arguments)
    else:
        result = f"Unknown tool: {function_name}"

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [38]:
messages = [
    {"role": "system", "content": instructions},
    {
        "role": "user",
        "content": "Add a documentation entry about testing agents."
    }
]

response = loop(messages)

print(response.output_text)

The documentation entry about testing agents has been successfully added. Here are the details:

**Title:** Testing Agents  
**Description:** A guide on how to test agents effectively in various scenarios.  
**Content:** 
- **Defining Test Criteria**: Set clear goals and metrics for success.
- **Creating Test Scenarios**: Develop scenarios the agent may encounter.
- **Simulating Environments**: Recreate operational conditions using simulation tools.
- **Executing Tests**: Run the agent through test scenarios and log results.
- **Analyzing Results**: Assess data to identify strengths and weaknesses.
- **Iterative Improvements**: Make adjustments based on outcomes and retest.
- **Documenting Findings**: Keep a detailed record of procedures and results.

**Filename:** testing_agents.md

Let me know if you need anything else!


In [41]:
class Agent:
    def __init__(self, client, model, instructions, tools):
        self.client = client
        self.model = model
        self.instructions = instructions
        self.tools = tools
        self.message_history = []

    def make_call(self, tool_call):
        function_name = tool_call.name
        arguments = json.loads(tool_call.arguments)

        if function_name == "search":
            result = search(**arguments)
        elif function_name == "add_entry":
            result = add_entry(**arguments)
        else:
            result = f"Unknown tool: {function_name}"

        return {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result),
        }

    def loop(self, messages):
        while True:
            response = self.client.responses.create(
                model=self.model,
                input=messages,
                tools=self.tools,
            )

            messages += response.output

            tool_calls = [
                item for item in response.output
                if item.type == "function_call"
            ]

            if not tool_calls:
                return response

            for tool_call in tool_calls:
                tool_result = self.make_call(tool_call)
                messages.append(tool_result)

    def qna(self, question):
        self.message_history.append(
            {"role": "user", "content": question}
        )

        messages = [
            {"role": "system", "content": self.instructions}
        ] + self.message_history

        response = self.loop(messages)

        self.message_history += response.output

        return response.output_text

In [42]:
agent = Agent(
    client=openai_client,
    model="gpt-4o-mini",
    instructions=instructions,
    tools=tools,
)

In [43]:
answer = agent.qna("How do I create a dashboard in Evidently?")

print(answer)

To create a dashboard in Evidently, you can follow these steps:

### 1. **Set Up Your Environment**
- Make sure you have created an **Evidently** account and **Project**. You can sign up for an account [here](https://app.evidently.cloud/signup).

### 2. **Run Evaluations**
- Before creating a dashboard, ensure that you run at least one evaluation and save a **Report** to your Project. This Report will serve as the data source for your dashboard.

### 3. **Dashboard Basics**
- A **Dashboard** provides a visual view of your AI application performance, enabling you to track evaluation results over time.

### 4. **Adding Panels**
   There are two ways to add panels to your dashboard:
   - **Using the Python API**: This method allows you to define your dashboard programmatically.
   - **Through the User Interface**: Available in Evidently Cloud and Enterprise, you can add panels directly from the dashboard interface.

#### **Using the API**
1. First, you need to import the necessary librari

In [44]:
answer = agent.qna(
    "What was the first thing you said I need before creating it?"
)

print(answer)

The first thing you need before creating a dashboard in Evidently is to set up your environment by creating an **Evidently** account and a **Project**.


In [45]:
from pydantic import BaseModel

class RAGResponse(BaseModel):
    answer: str
    found_answer: bool

In [46]:
from pydantic import BaseModel

class RAGResponse(BaseModel):
    answer: str
    found_answer: bool

In [47]:
response = openai_client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": instructions},
        {"role": "user", "content": "How do I create a dashboard in Evidently?"}
    ],
    tools=tools,
    text_format=RAGResponse,
)

In [48]:
response.output_parsed

In [49]:
response.output

[ParsedResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_27w0zRJj2JF1Zr0i3ybhsLQU', name='search', type='function_call', id='fc_0f975ddc5cacb695006ac7c71c949887d1b0acf54f72c1b5d8', async_=None, caller=None, namespace=None, status='completed', parsed_arguments=None)]

In [50]:
answer = agent.qna(
    "How do I create a dashboard in Evidently?"
)

print(answer)

To create a dashboard in Evidently, follow these steps:

### 1. **Set Up Your Environment**
- Create an **Evidently** account and a **Project**.

### 2. **Run Evaluations**
- Ensure you have run at least one evaluation and saved a **Report** to your Project. This Report will be the data source for your dashboard.

### 3. **Dashboard Basics**
- A **Dashboard** allows you to visually track your AI application performance over time.

### 4. **Adding Panels**
You can add panels using either:
- **Python API**: To define your dashboard programmatically.
- **User Interface**: Available in Evidently Cloud and Enterprise for direct addition of panels.

#### **Using the API**
1. Import necessary libraries:
   ```python
   from evidently.sdk.models import PanelMetric
   from evidently.sdk.panels import DashboardPanelPlot
   ```

2. **Add a Panel**:
   Example to add a counter panel showing `RowCount`:
   ```python
   project.dashboard.add_panel(
       DashboardPanelPlot(
           title="Row co

In [51]:
structured_response = openai_client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {
            "role": "system",
            "content": "Convert the answer into the required structured format."
        },
        {
            "role": "user",
            "content": answer
        }
    ],
    text_format=RAGResponse,
)

In [52]:
structured_response.output_parsed

RAGResponse(answer='To create a dashboard in Evidently, follow these steps:\n\n### 1. **Set Up Your Environment**\n- Create an **Evidently** account and a **Project**.\n\n### 2. **Run Evaluations**\n- Ensure you have run at least one evaluation and saved a **Report** to your Project. This Report will be the data source for your dashboard.\n\n### 3. **Dashboard Basics**\n- A **Dashboard** allows you to visually track your AI application performance over time.\n\n### 4. **Adding Panels**\nYou can add panels using either:\n- **Python API**: To define your dashboard programmatically.\n- **User Interface**: Available in Evidently Cloud and Enterprise for direct addition of panels.\n\n#### **Using the API**\n1. Import necessary libraries:\n   ```python\n   from evidently.sdk.models import PanelMetric\n   from evidently.sdk.panels import DashboardPanelPlot\n   ```\n\n2. **Add a Panel**:\n   Example to add a counter panel showing `RowCount`:\n   ```python\n   project.dashboard.add_panel(\n    

In [53]:
structure_result_tool = {
    "type": "function",
    "name": "structure_result",
    "description": "Return the final answer in a structured format.",
    "parameters": RAGResponse.model_json_schema(),
}

In [54]:
tools_with_structure = [
    search_tool,
    add_entry_tool,
    structure_result_tool,
]

In [55]:
def create_fake_tool(output_type, name="structure_result"):
    schema = output_type.model_json_schema()
    schema["type"] = "object"
    schema["additionalProperties"] = False

    structure_result_tool = {
        "type": "function",
        "name": name,
        "description": "Call this function when you're ready to show the final result as structured data.",
        "strict": True,
        "parameters": schema,
    }

    return structure_result_tool

In [56]:
class StructuredAgent2:

    FAKE_TOOL_NAME = "structure_result"

    def __init__(self, llm_client, model, instructions, tools, output_type):
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.tools = tools + [
            create_fake_tool(output_type, self.FAKE_TOOL_NAME)
        ]
        self.output_type = output_type

    def make_call(self, tool_call):
        arguments = json.loads(tool_call.arguments)
        name = tool_call.name

        if name == "search":
            result = search(**arguments)
        elif name == "add_entry":
            result = add_entry(**arguments)
        else:
            result = f'not found tool "{name}"'

        return {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result),
        }

    def loop(self, user_prompt, message_history=None):
        if not message_history:
            message_history = [
                {"role": "system", "content": self.instructions},
            ]

        message_history.append(
            {"role": "user", "content": user_prompt}
        )

        while True:
            response = self.llm_client.responses.create(
                model=self.model,
                input=message_history,
                tools=self.tools,
            )

            message_history.extend(response.output)

            output = None

            for message in response.output:
                if message.type == "function_call":

                    if message.name == self.FAKE_TOOL_NAME:
                        output_json = json.loads(message.arguments)
                        output = self.output_type.model_validate(
                            output_json
                        )
                        continue

                    tool_call_output = self.make_call(message)
                    message_history.append(tool_call_output)

            if output:
                break

        return message_history, output
        

In [57]:
structured_agent = StructuredAgent2(
    llm_client=openai_client,
    model="gpt-4o-mini",
    instructions=instructions,
    tools=[search_tool],
    output_type=RAGResponse,
)

In [58]:
message_history, output = structured_agent.loop(
    "How do I create a dashboard in Evidently?"
)

In [59]:
output

RAGResponse(answer='To create a dashboard in Evidently, follow these steps:\n\n1. **Create a Project**: Start by creating a project in Evidently after signing up and logging into your account.\n2. **Run an Evaluation**: Run an evaluation and save at least one report to your project. Each project has its own dashboard, which will be empty at first. \n3. **Add Panels**: You can add panels to the dashboard using:\n   - The **Python API** to define your dashboard as code.\n   - Through the **User Interface** (UI) for easier panel addition (Cloud and Enterprise versions only).\n\n4. **Customize Panels**: When adding panels, specify the:\n   - **Value**: Choose an individual metric to plot.\n   - **Parameters**: Set details like title, panel type, and size.\n   - **Tags** (optional): Use these to filter and visualize subsets of your data.\n\n5. **Organize Dashboard**: You can logically organize the panels into multiple tabs within the same dashboard.\n6. **View Reports**: Clicking any value 